# 📊 Telecom Consumption Intelligence - Feature Documentation

## 🎯 Project Overview
Realistic telecom user behavior simulation mimicking data patterns from carriers like **MTN, Vodafone, and Verizon**. The project uses a **two-dataset architecture** separating raw behavioral data from derived usage metrics - an industry best practice.

---

## 📁 Dataset Architecture

### Why TWO Datasets?
In real telecom systems, we separate:
1. **Raw Activity Data** - What users are *doing*
2. **Aggregated Usage Data** - How much data they *consumed*

This separation enables:
- ✅ SQL analytics on behavior patterns
- ✅ ML model retraining with new features
- ✅ Data lineage tracking
- ✅ Scalable architecture

---

## 📊 Dataset 1: `user_activity` (Raw Behavioral Data)

### 🎯 Purpose
Captures **user behavior patterns** - what users actually do on the network. This is the **feature store** for ML models.

### 📋 Feature Dictionary

| Feature | Type | Description | Business Context |
|---------|------|-------------|------------------|
| `user_id` | String | Unique subscriber ID (e.g., `SUB1234567`) | Primary key, anonymized IMSI equivalent |
| `hours_streaming` | Float (0-8) | Hours spent on video/music apps | **Highest data consumer** - Netflix, YouTube, Spotify |
| `hours_social` | Float (0-6) | Hours on social platforms | Moderate consumption - Instagram, TikTok, Facebook |
| `hours_messaging` | Float (0-4) | Hours on messaging apps | **Lowest consumption** - WhatsApp, Telegram |
| `hours_gaming` | Float (0-6) | Hours mobile gaming | Variable usage - latency sensitive |
| `background_data_mb` | Float (5-200) | System data consumption | Always-on - updates, sync, location |
| `is_peak_hour_user` | Binary (0/1) | Measurement during 6-9 PM | Network congestion indicator |
| `is_weekend` | Binary (0/1) | Weekend vs weekday | Usage pattern shift indicator |
| `age_group` | Categorical | User age bracket | Demographic segmentation |
| `plan_type` | Categorical | Subscription tier | Revenue segmentation, ARPU |
| `device_type` | Categorical | Device category | Hardware capability indicator |
| `network_type` | Categorical | Network technology (3G/4G/5G) | Infrastructure capability |
| `measurement_date` | Date | Timestamp of observation | Trend analysis |


## 💾 Dataset 2: `user_data_usage` (Derived Target Data)

### 🎯 Purpose
Contains **calculated data consumption** - this is what ML models **predict**. This dataset serves as the **target variable store** for supervised learning tasks.

**Key Concept:** While `user_activity` captures *behavior*, this dataset captures *outcomes* - the actual data consumed.

---

### 📋 Feature Dictionary

| Feature | Type | Description | Calculation |
|---------|------|-------------|-------------|
| `user_id` | String | Unique subscriber identifier | Links to `user_activity` table |
| `total_data_mb` | Float | **Primary ML Target** - Total daily data consumption in megabytes | Sum of all data sources + random variation |
| `streaming_data_mb` | Float | Data consumed from video/music streaming apps | `hours_streaming × streaming_rate` |
| `social_data_mb` | Float | Data consumed from social media platforms | `hours_social × social_rate` |
| `messaging_data_mb` | Float | Data consumed from messaging applications | `hours_messaging × messaging_rate` |
| `gaming_data_mb` | Float | Data consumed from mobile gaming | `hours_gaming × gaming_rate` |
| `data_usage_category` | Categorical | User classification based on consumption tier | Binned `total_data_gb` values |

---

### 📊 Data Consumption Rates (MB/hour)

| Activity | 3G Network | 4G Network | 5G Network | Business Notes |
|----------|------------|------------|------------|----------------|
| **Streaming** | 300 - 500 | 700 - 1,200 | 1,500 - 2,500 | Quality auto-adjusts to network speed |
| **Social Media** | 80 - 150 | 150 - 300 | 250 - 400 | Video autoplay significantly impacts usage |
| **Messaging** | 5 - 20 | 5 - 20 | 5 - 20 | Minimal network impact regardless of technology |
| **Gaming** | 40 - 200 | 40 - 200 | 40 - 200 | Varies by game type, not network speed |

> **Industry Insight:** Streaming dominates network traffic (~70-80% of total data), making it the primary driver of infrastructure costs.

---

### 🎯 Target Variable Details

#### Primary Target: `total_data_gb`
- **Use Case:** Regression models predicting data consumption



In [30]:
# IMPORT NECESSARY LIBRARIES
import os
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [31]:
# LOAD DATA

data_folder = r"G:\Study\DATA SCINCE\PROJECTS\POTFOLIO\telecom-consumption-intelligence\data\processed"
file_name = "user_activity_data_daily.csv"

try:
    # Read the CSV file
    df = pd.read_csv(f"{data_folder}\\{file_name}")
    print(f"Shape: {df.shape} (Rows: {df.shape[0]}, Columns: {df.shape[1]})")
    print("Data loaded successfully.")
    display(df.head())

    # Display basic info about the dataframe
    print('\n DataFrame Info:')
    display(df.info())

    # missing values summary
    print("\n Missing Values:")
    missing = df.isna().sum().to_frame(name='Missing Values')
    missing['missing %'] = (missing['Missing Values'] / len(df))*100
    display(missing.sort_values(by='missing %', ascending=False))

except FileNotFoundError:
    print("Csv file not found. Please check the file path.")    
except Exception as e:
    print(f"Error reading Csv file: {e}")

Shape: (10006, 18) (Rows: 10006, Columns: 18)
Data loaded successfully.


,user_id,measurement_date,hours_streaming,hours_social,hours_messaging,hours_gaming,streaming_data_mb,social_data_mb,messaging_data_mb,gaming_data_mb,total_data_mb,is_peak_hour_user,is_weekend,age_group,plan_type,device_type,network_type,data_usage_category
0,SUB7423388,2026-04-07,0.33,0.66,0.87,0.37,542.92,133.07,13.32,60.29,756.41,0,1,18-24,Prepaid_Monthly,Mid_Range,4G+,Light
1,SUB7550634,2026-04-06,1.07,3.03,0.46,0.04,1214.57,814.33,8.67,3.00,2292.13,0,0,45-54,Postpaid_Premium,5G_Device,4G,Moderate
2,SUB5304572,2026-03-20,0.54,0.45,0.71,1.25,814.07,102.50,11.07,149.01,1124.30,1,0,25-34,Postpaid_Unlimited,Mid_Range,4G,Light
3,SUB3234489,2026-04-14,3.38,1.92,0.00,1.18,12057.73,624.75,0.00,132.93,12649.53,1,0,45-54,Postpaid_Unlimited,5G_Device,5G,Power_User
4,SUB8204212,2026-03-31,0.25,0.38,0.13,0.47,178.69,92.37,0.86,71.80,500.87,0,1,25-34,Prepaid_Daily,Premium_Smartphone,4G,Very_Light



 DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10006 entries, 0 to 10005
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              10006 non-null  object 
 1   measurement_date     10006 non-null  object 
 2   hours_streaming      10006 non-null  float64
 3   hours_social         10006 non-null  float64
 4   hours_messaging      10006 non-null  float64
 5   hours_gaming         10006 non-null  float64
 6   streaming_data_mb    10006 non-null  float64
 7   social_data_mb       10006 non-null  float64
 8   messaging_data_mb    10006 non-null  float64
 9   gaming_data_mb       10006 non-null  float64
 10  total_data_mb        10006 non-null  float64
 11  is_peak_hour_user    10006 non-null  int64  
 12  is_weekend           10006 non-null  int64  
 13  age_group            10006 non-null  object 
 14  plan_type            10006 non-null  object 
 15  device_type       

None


 Missing Values:


,Missing Values,missing %
user_id,0,0.00
measurement_date,0,0.00
hours_streaming,0,0.00
hours_social,0,0.00
hours_messaging,0,0.00
hours_gaming,0,0.00
streaming_data_mb,0,0.00
social_data_mb,0,0.00
messaging_data_mb,0,0.00
gaming_data_mb,0,0.00


In [32]:
def comprehensive_describe(df):
    """Generate comprehensive descriptive statistics"""
    if df is None:
        print("Error: DataFrame is None.")
        return
    
    # Basic info
    print("="*60)
    print("DATAFRAME OVERVIEW")
    print("="*60)
    print(f"Data Types:\n{df.dtypes.value_counts()}")
    
    # Numeric columns
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    if len(numeric_cols) > 0:
        print("\n" + "="*60)
        print(f"NUMERIC COLUMNS ({len(numeric_cols)}):")
        print("="*60)
        display(df[numeric_cols].describe().T)
    
    # Categorical/Text columns
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    if len(cat_cols) > 0:
        print("\n" + "="*60)
        print(f"CATEGORICAL/TEXT COLUMNS ({len(cat_cols)}):")
        print("="*60)
        display(df[cat_cols].describe(include='O').T)
        
        # Show unique value counts for categorical columns
        print("\nUnique Values per Categorical Column:")
        for col in cat_cols:
            print(f"  {col}: {df[col].nunique()} unique values")

# Usage
comprehensive_describe(df)

DATAFRAME OVERVIEW
Data Types:
float64    9
object     7
int64      2
Name: count, dtype: int64

NUMERIC COLUMNS (11):


,count,mean,std,min,25%,50%,75%,max
hours_streaming,10006.00,1.70,1.84,0.00,0.44,1.05,2.23,8.00
hours_social,10006.00,1.65,1.55,0.00,0.52,1.13,2.26,6.00
hours_messaging,10006.00,0.58,0.76,0.00,0.11,0.30,0.73,4.00
hours_gaming,10006.00,0.59,0.89,0.00,0.09,0.27,0.71,6.00
streaming_data_mb,10006.00,2738.07,4000.00,0.00,458.52,1261.22,3234.66,34221.24
social_data_mb,10006.00,417.70,447.73,0.00,109.49,251.69,555.55,2395.45
messaging_data_mb,10006.00,7.31,10.49,0.00,1.21,3.51,8.73,79.83
gaming_data_mb,10006.00,70.96,118.77,0.00,9.07,29.67,80.01,1187.91
total_data_mb,10006.00,3311.64,4263.09,11.50,800.75,1784.66,3958.67,37874.04
is_peak_hour_user,10006.00,0.29,0.46,0.00,0.00,0.00,1.00,1.00



CATEGORICAL/TEXT COLUMNS (7):


,count,unique,top,freq
user_id,10006,9997,SUB8156406,4
measurement_date,10006,30,2026-04-02,369
age_group,10006,5,25-34,3049
plan_type,10006,5,Prepaid_Monthly,2548
device_type,10006,5,Mid_Range,3517
network_type,10006,4,4G,4480
data_usage_category,10006,5,Light,3977



Unique Values per Categorical Column:
  user_id: 9997 unique values
  measurement_date: 30 unique values
  age_group: 5 unique values
  plan_type: 5 unique values
  device_type: 5 unique values
  network_type: 4 unique values
  data_usage_category: 5 unique values


## Notes on Data Scaling

## MB to GB Conversion
We need to change the scale from **MB** to **GB** because our primary KPI (`total_data_gb`) is measured in GB.

### Columns to Convert:
- `streaming_data_mb`
- `social_data_mb`
- `messaging_data_mb`
- `gaming_data_mb`
- `total_data_mb`

### Rationale:
- Ensures consistency with the primary target variable.
- Aligns with industry standards for data consumption reporting (e.g., GB for user plans).
- Improves readability and interpretation in visualizations and models.

In [33]:
def convert_mb_to_gb(df, mb_columns=None, drop_original=True):  # Changed default to True
    """
    Convert MB to GB using telecom standard (1 GB = 1000 MB).
    Drops original MB columns by default.
    """
    if mb_columns is None:
        mb_columns = [col for col in df.columns if col.endswith('_mb')]
    
    for col in mb_columns:
        new_col = col.replace('_mb', '_gb')
        df[new_col] = df[col] / 1000  # Telecom standard
        print(f"Converted '{col}' → '{new_col}' (÷ 1000)")
    
    if drop_original:
        df = df.drop(columns=mb_columns)
        print(f"Dropped {len(mb_columns)} original MB columns")
    
    return df


df = convert_mb_to_gb(df)

Converted 'streaming_data_mb' → 'streaming_data_gb' (÷ 1000)
Converted 'social_data_mb' → 'social_data_gb' (÷ 1000)
Converted 'messaging_data_mb' → 'messaging_data_gb' (÷ 1000)
Converted 'gaming_data_mb' → 'gaming_data_gb' (÷ 1000)
Converted 'total_data_mb' → 'total_data_gb' (÷ 1000)
Dropped 5 original MB columns


In [34]:
df['measurement_date'] = pd.to_datetime(df['measurement_date'], errors='coerce')
print(f"Converted 'measurement_date' to datetime. Null values after conversion: {df['measurement_date'].isnull().sum()}")
print(f"Data types after conversion:\n{df.dtypes}")

Converted 'measurement_date' to datetime. Null values after conversion: 0
Data types after conversion:
user_id                        object
measurement_date       datetime64[ns]
hours_streaming               float64
hours_social                  float64
hours_messaging               float64
hours_gaming                  float64
is_peak_hour_user               int64
is_weekend                      int64
age_group                      object
plan_type                      object
device_type                    object
network_type                   object
data_usage_category            object
streaming_data_gb             float64
social_data_gb                float64
messaging_data_gb             float64
gaming_data_gb                float64
total_data_gb                 float64
dtype: object


In [35]:

STAGE_NAME = "curated" 

# Define paths
base_path = Path(r"G:\Study\DATA SCINCE\PROJECTS\POTFOLIO\telecom-consumption-intelligence\data")
curated_path = base_path / STAGE_NAME
curated_path.mkdir(parents=True, exist_ok=True)
# Create EDA folder 
Path(eda_folder).mkdir(parents=True, exist_ok=True)

# ============================================
# SAVE MULTIPLE VERSIONS
# ============================================

# 1. PARQUET 
df.to_parquet(f"{curated_path}\\user_activity_curated.parquet", index=False)
print("✓ Saved as Parquet (fast, compact, preserves dtypes)")

# 2. CSV (Backup/Compatibility)
df.to_csv(f"{curated_path}\\user_activity_curated.csv", index=False)
print("✓ Saved as CSV (universal compatibility)")


# ============================================
# SAVE METADATA
# ============================================

# Save data dictionary/metadata
metadata = {
    'preparation_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'shape': df.shape,
    'columns': df.columns.tolist(),
    'dtypes': df.dtypes.to_dict(),
    'target_variable': 'total_data_gb',
    'transformations_applied': [
        'Converted MB columns to GB (÷ 1000)',
        'Removed original MB columns',
        'Any other transformations you did...'
    ],
    'columns_converted': {
        'streaming_data_mb → streaming_data_gb',
        'social_data_mb → social_data_gb',
        'messaging_data_mb → messaging_data_gb',
        'gaming_data_mb → gaming_data_gb',
        'total_data_mb → total_data_gb'
    }
}

# Save metadata as JSON
import json
with open(f"{curated_path}\\data_preparation_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=4, default=str)
print("✓ Saved preparation metadata")

✓ Saved as Parquet (fast, compact, preserves dtypes)
✓ Saved as CSV (universal compatibility)
✓ Saved preparation metadata
